# LLM-Based GORE Pipeline — Execution

This notebook executes the complete Goal-Oriented Requirements Engineering pipeline. It separates the initial top-down generation from the proposed iterative bottom-up extension, while reusing the same evaluated low-level-goal generator whenever a decomposition must be created again.

The execution flow is:

1. environment and dataset configuration;
2. evaluated top-down extraction of actors, high-level goals, and low-level goals;
3. bounded bottom-up reconstruction and global evaluation;
4. selective top-down regeneration of low-level goals when a branch changes or a missing high-level goal is added;
5. restart of bottom-up reconstruction and re-evaluation on the updated hierarchy;
6. persistence of all intermediate and final artifacts;
7. goal-to-API alignment using the final verified low-level goals.

In [16]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

from key import (
    get_key_openai,
    get_key_llama,
    count_Llama_keys,
)

openai_key = get_key_openai()
groq_key = get_key_llama()

print("Chiave OpenAI presente:", bool(openai_key))
print("Formato OpenAI plausibile:", openai_key.startswith("sk-"))

print("Chiave Groq presente:", bool(groq_key))
print("Formato Groq plausibile:", groq_key.startswith("gsk_"))

print("Numero chiavi Groq:", count_Llama_keys())

Project root: c:\Users\agnes_lryeu3v\Desktop\tesi
Chiave OpenAI presente: True
Formato OpenAI plausibile: True
Chiave Groq presente: True
Formato Groq plausibile: True
Numero chiavi Groq: 5


## 1. Environment and configuration

This section configures the project path, imports the pipeline components, selects the prompting mode, enables or disables the LLaMA ablation, and loads the datasets used during execution.


In [17]:
from pathlib import Path
import time
import json
import os
import sys

# Resolve the project root whether the notebook is launched from the project
# directory or from a notebooks/ subdirectory.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from groundtruth import (
    GENOME,
    GESTAO_HOSPITAL,
    SIA_PROJECT_25_26,
    LONDON_AMBULANCE_SYSTEM,
)
from src.extraction.extractor import (
    generate_description,
    generate_actors,
    generate_high_level_goals,
    generate_low_level_goals,
)
from src.mapping.APIs_mapping import (
    generate_mapping_apis_goals,
    print_api_goal_mapping,
)
from src.self_critique.refine_response import (
    EvalMode,
    generate_response_with_reflection,
)
from src.utils import get_api_list_from_swagger
from src.examples.shot_learning import ShotPromptingMode
#GENOME,
#GESTAO_HOSPITAL,
#LONDON_AMBULANCE_SYSTEM,

GROUNDTRUTHS = [
    SIA_PROJECT_25_26,
]

LLAMA_ABLATION = False
PROMPTING_MODE = ShotPromptingMode.FEW_SHOT
OUTPUT_PATH = PROJECT_ROOT / "output"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)


def output_file_for(dataset_name: str) -> Path:
    suffix = "_noLlama" if LLAMA_ABLATION else ""
    return OUTPUT_PATH / f"{dataset_name}_{PROMPTING_MODE.name}{suffix}.json"


## 2. Baseline top-down pipeline

The baseline architecture follows a top-down process. Starting from the textual documentation, it extracts actors, identifies high-level goals associated with those actors, and decomposes each high-level goal into operational low-level goals. Each extraction stage is executed through the existing reflection-based refinement mechanism.

The original architecture is kept conceptually separate from the proposed extension so that its outputs can be reused as a baseline during the experimental evaluation.


### 2.1 Actor extraction

The actor extractor identifies the entities that interact with the system or pursue goals through it. The reflection loop evaluates and, when required, refines the generated actor set.

### 2.2 High-level goal extraction

The high-level goal extractor derives abstract stakeholder or system intentions from the documentation and associates them with the previously extracted actors.

### 2.3 Low-level goal extraction

The low-level goal extractor decomposes the high-level goals into concrete and operational objectives. These definitive low-level goals become the input of the proposed bottom-up reconstruction stage.


In [9]:
generated_descriptions = {}
generated_actors = {}
generated_hl = {}
generated_ll = {}
generated_actor_objects = {}
generated_hl_objects = {}
generated_ll_objects = {}
baseline_errors = {}


def generate_evaluated_low_level_goals(
    high_level_goals,
    description,
    actors,
):
    """Run the existing top-down LLG generator with its evaluator.

    The same function is used both for the initial decomposition and for every
    selective regeneration requested by the global feedback cycle.
    """
    low_level_goals, score, critique = generate_response_with_reflection(
        "Low Level Goals",
        generate_low_level_goals,
        define_args=(high_level_goals,),
        eval_mode=EvalMode.LOW_LEVEL,
        eval_args=(description, actors, high_level_goals),
        shotPromptingMode=PROMPTING_MODE,
        llama_ablation=LLAMA_ABLATION,
    )
    return low_level_goals, score, critique


def run_baseline_for_dataset(groundtruth: dict) -> None:
    dataset_name = groundtruth["name"]

    try:
        description = (
            str(generate_description(groundtruth["link-readme"]))
            if "link-readme" in groundtruth
            else groundtruth["description"]
        )
        generated_descriptions[dataset_name] = description

        actors, actors_score, actors_critique = generate_response_with_reflection(
            "Actors",
            generate_actors,
            define_args=(description,),
            eval_mode=EvalMode.ACTORS,
            eval_args=(description,),
            shotPromptingMode=PROMPTING_MODE,
            llama_ablation=LLAMA_ABLATION,
        )

        generated_actor_objects[dataset_name] = actors
        generated_actors[dataset_name] = [actor.name for actor in actors.actors]

        high_level_goals, hl_score, hl_critique = generate_response_with_reflection(
            "High Level Goals",
            generate_high_level_goals,
            define_args=(description, actors),
            eval_mode=EvalMode.HIGH_LEVEL,
            eval_args=(description, actors),
            shotPromptingMode=PROMPTING_MODE,
            llama_ablation=LLAMA_ABLATION,
        )

        generated_hl_objects[dataset_name] = high_level_goals
        generated_hl[dataset_name] = [
            goal.description for goal in high_level_goals.goals
        ]

        low_level_goals, ll_score, ll_critique = (
            generate_evaluated_low_level_goals(
                high_level_goals=high_level_goals,
                description=description,
                actors=actors,
            )
        )

        generated_ll_objects[dataset_name] = low_level_goals
        generated_ll[dataset_name] = [
            goal.description for goal in low_level_goals.low_level_goals
        ]

        output = {
            "name": dataset_name,
            "description": description,
            "actors": generated_actors[dataset_name],
            "highLevelGoals": generated_hl[dataset_name],
            "lowLevelGoals": generated_ll[dataset_name],
            "baselineScores": {
                "actors": actors_score,
                "highLevelGoals": hl_score,
                "lowLevelGoals": ll_score,
            },
            "baselineCritiques": {
                "actors": actors_critique,
                "highLevelGoals": hl_critique,
                "lowLevelGoals": ll_critique,
            },
        }

        with output_file_for(dataset_name).open("w", encoding="utf-8") as file:
            json.dump(output, file, indent=4, ensure_ascii=False, default=str)

        print(f"[{dataset_name}] Baseline extraction completed.")

    except Exception as error:
        baseline_errors[dataset_name] = f"{type(error).__name__}: {error}"
        print(f"[{dataset_name}] Baseline extraction failed: {error}")


threads = []
for groundtruth in GROUNDTRUTHS:
    thread = Thread(target=run_baseline_for_dataset, args=(groundtruth,))
    thread.start()
    threads.append(thread)

for thread in threads:
    thread.join()

print(f"Completed datasets: {len(generated_ll_objects)} / {len(GROUNDTRUTHS)}")
if baseline_errors:
    print("Baseline errors:", baseline_errors)

Actors STARTING... (attempt 1)
No feedback provided!
Actors DONE...
actors=[Actor(name='Citizens', description='Individuals who report issues and malfunctions in the urban environment.'), Actor(name='Municipal Operators', description='Staff members of the Municipality of Turin who review and manage citizen reports.'), Actor(name='External Maintenance Personnel', description='Workers from external companies who address specific issues reported by citizens.'), Actor(name='Municipal Administrators', description='Officials who configure the system and oversee the assignment of reports.'), Actor(name='Non-registered Users', description='Individuals who can view reports and statistics without needing to register.')]
High Level Goals STARTING... (attempt 1)
No feedback provided!
This is the provided sys prompt:  You are a helpful assistant expert in software engineering tasks.You're tasked to extract high level goals from a software description for each provided actor that is expected to inte

## 3. Proposed iterative bottom-up extension

The extension starts from the evaluated top-down hierarchy. For each iteration, `run_global_goal_cycle` reconstructs a candidate high-level goal from every branch, evaluates it globally, and updates only the parts of the hierarchy that require revision.

The cycle also performs a global documentation-coverage check after all current branches are confirmed. When the evaluator identifies a functional intention that is present in the documentation but absent from the current high-level-goal collection, the new high-level goal is appended. The pipeline then returns to the preceding top-down step: low-level goals are generated and evaluated for the new high-level goal. After this selective top-down generation, the bottom-up reconstruction and global evaluation are executed again on the updated hierarchy.

Existing confirmed branches are preserved; only revised branches and newly added high-level goals receive a new low-level decomposition.

### 3.1 Complete refinement–abstraction–verification cycle

The cycle receives the definitive baseline `HighLevelGoals` and `LowLevelGoals` objects. The same reflection-based top-down low-level generator used in the baseline is injected as a callback.

At each iteration the orchestrator:

1. reconstructs each branch bottom-up from its current low-level goals;
2. evaluates the reconstructed goal against its parent, all existing high-level goals, and the documentation;
3. updates the high-level-goal collection when required;
4. selectively regenerates and evaluates low-level goals for changed or newly introduced high-level goals;
5. starts a new bottom-up iteration using the updated hierarchy;
6. once all branches are confirmed, checks whether the documentation still contains missing high-level goals.

Each iteration is persisted through `GlobalGoalCycleIteration`, including traceability, decisions, added goals, regenerated decompositions, errors, convergence status, and the deterministic state signature.

In [ ]:
from src.bottom_up.goal_cycle_orchestrator import run_global_goal_cycle

MAX_GLOBAL_CYCLE_ITERATIONS = 2

global_cycle_results = {}
global_cycle_errors = {}
final_high_level_goals_objects = {}
final_low_level_goals_objects = {}

for dataset_name, initial_high_level_goals in generated_hl_objects.items():
    if dataset_name not in generated_ll_objects:
        global_cycle_errors[dataset_name] = (
            "Initial low-level goals are not available."
        )
        print(
            f"[{dataset_name}] Global goal cycle skipped: "
            "initial low-level goals are not available."
        )
        continue

    initial_low_level_goals = generated_ll_objects[dataset_name]
    description = generated_descriptions[dataset_name]
    actors = generated_actor_objects[dataset_name]

    def regenerate_low_level_goals_for_cycle(
        high_level_goals,
        _description=description,
        _actors=actors,
    ):
        regenerated, _score, _critique = (
            generate_evaluated_low_level_goals(
                high_level_goals=high_level_goals,
                description=_description,
                actors=_actors,
            )
        )
        return regenerated

    try:
        cycle_result = run_global_goal_cycle(
            project_description=description,
            initial_high_level_goals=initial_high_level_goals,
            initial_low_level_goals=initial_low_level_goals,
            regenerate_low_level_goals=(
                regenerate_low_level_goals_for_cycle
            ),
            max_iterations=MAX_GLOBAL_CYCLE_ITERATIONS,
        )
    except Exception as error:
        global_cycle_errors[dataset_name] = (
            f"{type(error).__name__}: {error}"
        )
        print(
            f"[{dataset_name}] Global goal cycle failed: "
            f"{type(error).__name__}: {error}"
        )
        continue

    global_cycle_results[dataset_name] = cycle_result
    final_high_level_goals_objects[dataset_name] = (
        cycle_result.final_high_level_goals
    )
    final_low_level_goals_objects[dataset_name] = (
        cycle_result.final_low_level_goals
    )

    output_file = output_file_for(dataset_name)
    with output_file.open("r", encoding="utf-8") as file:
        output = json.load(file)

    output["globalGoalCycle"] = cycle_result.model_dump(
        mode="json"
    )
    output["finalHighLevelGoals"] = (
        cycle_result.final_high_level_goals.model_dump(
            mode="json"
        )
    )
    output["finalLowLevelGoals"] = (
        cycle_result.final_low_level_goals.model_dump(
            mode="json"
        )
    )

    with output_file.open("w", encoding="utf-8") as file:
        json.dump(
            output,
            file,
            indent=4,
            ensure_ascii=False,
        )

    print(
        f"[{dataset_name}] Global cycle completed: "
        f"converged={cycle_result.converged}, "
        f"stop_reason={cycle_result.stop_reason}, "
        f"iterations={cycle_result.completed_iterations}."
    )

SyntaxError: no binding for nonlocal 'regeneration_call_counter' found (2972704738.py, line 95)

### 3.2 Cycle result inspection

The following cell prints a compact per-dataset summary. Detailed iteration artifacts are already persisted under `globalGoalCycle` in the corresponding JSON output.

In [ ]:
for dataset_name, cycle_result in global_cycle_results.items():
    print()
    print(f"[{dataset_name}]")
    print(f"  Converged: {cycle_result.converged}")
    print(f"  Stop reason: {cycle_result.stop_reason}")
    print(
        f"  Completed iterations: "
        f"{cycle_result.completed_iterations}"
    )
    print(
        f"  Final high-level goals: "
        f"{len(cycle_result.final_high_level_goals.goals)}"
    )
    print(
        f"  Final low-level goals: "
        f"{len(cycle_result.final_low_level_goals.low_level_goals)}"
    )
    print(
        f"  High-level goals added during the cycle: "
        f"{len(cycle_result.added_high_level_goals)}"
    )
    for added_goal in cycle_result.added_high_level_goals:
        print(f"    + {added_goal.name}")

    if cycle_result.unresolved_bottom_up_errors:
        print(
            "  Unresolved bottom-up errors:",
            cycle_result.unresolved_bottom_up_errors,
        )

    if cycle_result.unresolved_global_evaluation_errors:
        print(
            "  Unresolved global-evaluation errors:",
            cycle_result.unresolved_global_evaluation_errors,
        )

    if cycle_result.unresolved_empty_branches:
        print(
            "  Unresolved empty branches:",
            cycle_result.unresolved_empty_branches,
        )

if global_cycle_errors:
    print()
    print("Global cycle execution errors:", global_cycle_errors)

### 3.3 Final goal collections

The final collections are taken directly from `GlobalGoalCycleResult`. They therefore include high-level goals added by the branch evaluator or by the documentation-coverage evaluator, together with the low-level goals generated through the same evaluated top-down mechanism. A newly added high-level goal is not considered final until its decomposition has entered a subsequent bottom-up reconstruction and global re-evaluation, unless the maximum-iteration bound is reached first.

In [ ]:
final_high_level_goals = {
    dataset_name: [
        goal.description
        for goal in goals.goals
    ]
    for dataset_name, goals in final_high_level_goals_objects.items()
}

final_low_level_goals = {
    dataset_name: [
        goal.description
        for goal in goals.low_level_goals
    ]
    for dataset_name, goals in final_low_level_goals_objects.items()
}

for dataset_name in final_low_level_goals_objects:
    print(
        f"[{dataset_name}] Final goal collections available: "
        f"{len(final_high_level_goals_objects[dataset_name].goals)} HLG, "
        f"{len(final_low_level_goals_objects[dataset_name].low_level_goals)} LLG."
    )

## 4. Output persistence and traceability

Each dataset JSON contains the baseline artifacts and the complete iterative-extension result:

- `actors`;
- `highLevelGoals`;
- `lowLevelGoals`;
- `baselineScores`;
- `baselineCritiques`;
- `globalGoalCycle`;
- `finalHighLevelGoals`;
- `finalLowLevelGoals`.

`globalGoalCycle` contains the convergence status, stop reason, final collections, unresolved errors, and the complete ordered trace of all iterations. The traceability metadata is persisted for analysis but is never exposed to the bottom-up generator.

In [ ]:
for groundtruth in GROUNDTRUTHS:
    dataset_name = groundtruth["name"]
    output_file = output_file_for(dataset_name)

    if not output_file.exists():
        print(f"[{dataset_name}] Output file not available.")
        continue

    with output_file.open("r", encoding="utf-8") as file:
        output = json.load(file)

    print(
        f"[{dataset_name}] Saved sections: "
        f"{', '.join(output.keys())}"
    )


## 5. Goal-to-API alignment

The API alignment stage remains part of the original architecture. It is executed after the iterative extension and consumes `finalLowLevelGoals`, namely the last fully evaluated low-level decomposition returned by the cycle.

### 5.1 API extraction from Swagger

For each dataset that provides a Swagger source, this block extracts the available APIs.


In [ ]:
api_lists = {}

for groundtruth in GROUNDTRUTHS:
    dataset_name = groundtruth["name"]

    if "swagger" not in groundtruth:
        print(f"[{dataset_name}] No Swagger source configured; skipping API extraction.")
        continue

    print(f"[{dataset_name}] API extraction started...")
    api_lists[dataset_name] = get_api_list_from_swagger(
        link=groundtruth["swagger"]
    )
    print(
        f"[{dataset_name}] API extraction completed: "
        f"{len(api_lists[dataset_name])} API(s)."
    )


### 5.2 API mapping to final low-level goals

Each extracted API list is mapped to the final low-level goals returned by the global cycle. Datasets for which the cycle could not produce a final result are skipped and recorded separately.

In [ ]:
api_mappings = {}
api_mapping_skipped_due_to_cycle_errors = {}

for dataset_name, api_list in api_lists.items():
    if dataset_name not in final_low_level_goals_objects:
        error = global_cycle_errors.get(
            dataset_name,
            "Final low-level goals are not available.",
        )
        api_mapping_skipped_due_to_cycle_errors[dataset_name] = error
        print(f"[{dataset_name}] API mapping skipped: {error}")
        continue

    low_level_goals_for_mapping = (
        final_low_level_goals_objects[dataset_name]
    )

    print(f"[{dataset_name}] API mapping started...")
    mappings = generate_mapping_apis_goals(
        low_level_goals_for_mapping,
        api_list,
    )
    api_mappings[dataset_name] = mappings

    print_api_goal_mapping(mappings)

    mapping_file = OUTPUT_PATH / (
        f"final_mapping_{dataset_name}_{PROMPTING_MODE.name}"
        f"{'_noLlama' if LLAMA_ABLATION else ''}.json"
    )

    with mapping_file.open("w", encoding="utf-8") as file:
        json.dump(
            [mapping.model_dump(mode="json") for mapping in mappings],
            file,
            indent=4,
            ensure_ascii=False,
        )

    print(f"[{dataset_name}] API mapping saved to {mapping_file}.")

## 6. Execution summary

This notebook intentionally does not compute precision, recall, F1, semantic-similarity curves, or LLM-as-a-judge comparison statistics. Those analyses should read the persisted JSON files from a separate experimental-evaluation notebook, ensuring that generation and evaluation remain reproducible and independently repeatable.


In [ ]:
print("Execution summary")
print("-----------------")
print(f"Configured datasets: {len(GROUNDTRUTHS)}")
print(f"Baseline completed: {len(generated_ll_objects)}")
print(f"Global cycles completed: {len(global_cycle_results)}")
print(
    "Converged global cycles: "
    f"{sum(1 for result in global_cycle_results.values() if result.converged)}"
)
print(
    "Cycles stopped without convergence: "
    f"{sum(1 for result in global_cycle_results.values() if not result.converged)}"
)
print(f"Final goal collections available: {len(final_low_level_goals_objects)}")
print(f"API mappings completed: {len(api_mappings)}")
print(f"Global cycle execution errors: {len(global_cycle_errors)}")
print(
    "API mappings skipped due to cycle errors: "
    f"{len(api_mapping_skipped_due_to_cycle_errors)}"
)

if baseline_errors:
    print("Baseline errors:", baseline_errors)

if global_cycle_errors:
    print("Global cycle errors:", global_cycle_errors)

## 7. Standalone bottom-up run (baseline comparison)

This section runs **only** the bottom-up extension, starting from a top-down baseline output already saved on disk under `output/` (produced by section 2 in a previous run). It does not re-run actor/high-level/low-level goal extraction.

Purpose: let you re-run just the bottom-up cycle on an existing baseline and compare its final goal collections against the original top-down-only output, to judge whether the bottom-up structure improves the result.

Steps:
1. pick an existing baseline JSON from `output/`;
2. group its flat low-level goals under their existing high-level parents using `src/bottom_up/low_level_goal_mapper.py` (deterministic, no LLM call);
3. run `run_global_goal_cycle` (the same bottom-up orchestrator used in section 3) on the loaded hierarchy;
4. compare the baseline high-level/low-level goals against the ones produced after the bottom-up cycle, including the per-branch decisions (`CONFIRM_BRANCH`, `REGENERATE_LOW_LEVEL_GOALS`, `MATCHES_OTHER_HIGH_LEVEL_GOAL`, `ADD_NEW_HIGH_LEVEL_GOAL`, `REWRITE_ORIGINAL_HIGH_LEVEL_GOAL`).

Requirement: only section 1 (API keys) needs to have been run before this section.

In [29]:
from pathlib import Path
import json
import sys

# Self-contained setup: only requires the API keys loaded in section 1.
# It does not depend on the baseline-generation cells in section 2, so this
# whole section can be run on its own against an already-saved output file.
STANDALONE_PROJECT_ROOT = Path.cwd()
if not (STANDALONE_PROJECT_ROOT / "src").exists() and (
    STANDALONE_PROJECT_ROOT.parent / "src"
).exists():
    STANDALONE_PROJECT_ROOT = STANDALONE_PROJECT_ROOT.parent

if str(STANDALONE_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(STANDALONE_PROJECT_ROOT))

OUTPUT_PATH = STANDALONE_PROJECT_ROOT / "output"

# Existing top-down baseline JSON to use as input for the standalone
# bottom-up run. Change this to point at a different saved output file.
baseline_output_file = Path(
    "output/SIA Project 25 26_ONE_SHOT.json"
)
assert baseline_output_file.exists(), (
    f"Baseline file not found: {baseline_output_file}. "
    "Run section 2 first, or point baseline_output_file at an existing "
    "output/*.json file."
)

bottom_up_input_file = (
    OUTPUT_PATH / "bottom_up_inputs" / f"{baseline_output_file.stem}_bottom_up_input.json"
)
standalone_iterations_dir = OUTPUT_PATH / "bottom_up_iterations" / baseline_output_file.stem

print("Baseline file:", baseline_output_file)

Baseline file: output\SIA Project 25 26_ONE_SHOT.json


In [30]:
from src.bottom_up.low_level_goal_mapper import map_low_level_goals, load_mapped_bottom_up_input

# Groups the existing (flat) low-level goals under their existing high-level
# parents, using the deterministic BRANCH_SPECIFICATIONS registered for this
# dataset/prompting-mode/ablation combination. No LLM call happens here.
mapped_payload = map_low_level_goals(
    source_file=baseline_output_file,
    destination_file=bottom_up_input_file,
)
standalone_source_metadata = mapped_payload["source"]

(
    standalone_project_description,
    standalone_initial_high_level_goals,
    standalone_initial_low_level_goals,
) = load_mapped_bottom_up_input(bottom_up_input_file)

print(f"Dataset: {standalone_source_metadata['dataset_name']}")
print(f"Prompting mode: {standalone_source_metadata['prompting_mode']}")
print(f"No-LLaMA ablation: {standalone_source_metadata['no_llama']}")
print(f"Initial high-level goals: {len(standalone_initial_high_level_goals.goals)}")
print(f"Initial low-level goals: {len(standalone_initial_low_level_goals.low_level_goals)}")

Dataset: SIA Project 25 26
Prompting mode: ONE_SHOT
No-LLaMA ablation: False
Initial high-level goals: 5
Initial low-level goals: 17


In [31]:
from src.data_model import Actors
from src.extraction.extractor import generate_low_level_goals
from src.self_critique.refine_response import EvalMode, generate_response_with_reflection
from src.examples.shot_learning import ShotPromptingMode

# Must match the prompting mode / ablation used to produce the baseline file
# selected above, so that regeneration follows the same conventions.
STANDALONE_PROMPTING_MODE = ShotPromptingMode[standalone_source_metadata["prompting_mode"]]
STANDALONE_LLAMA_ABLATION = standalone_source_metadata["no_llama"]

# The baseline JSON only stores actor names, not full Actor objects, so the
# actors are rebuilt here as the unique set already attached to the mapped
# high-level goals (in source order, without duplicates).
seen_actor_names = set()
standalone_actors_list = []
for goal in standalone_initial_high_level_goals.goals:
    if goal.actor.name not in seen_actor_names:
        seen_actor_names.add(goal.actor.name)
        standalone_actors_list.append(goal.actor)
standalone_actors = Actors(actors=standalone_actors_list)


def regenerate_low_level_goals_for_standalone_cycle(high_level_goals):
    """Callback injected into the bottom-up cycle: regenerates and evaluates
    the low-level goals for the given high-level goals, reusing the same
    reflection-based top-down generator as the baseline (section 2)."""
    regenerated, _score, _critique = generate_response_with_reflection(
        "Low Level Goals",
        generate_low_level_goals,
        define_args=(high_level_goals,),
        eval_mode=EvalMode.LOW_LEVEL,
        eval_args=(standalone_project_description, standalone_actors, high_level_goals),
        shotPromptingMode=STANDALONE_PROMPTING_MODE,
        llama_ablation=STANDALONE_LLAMA_ABLATION,
    )
    return regenerated

In [32]:
from src.bottom_up.goal_cycle_orchestrator import run_global_goal_cycle

STANDALONE_MAX_ITERATIONS = 2

standalone_cycle_result = run_global_goal_cycle(
    project_description=standalone_project_description,
    initial_high_level_goals=standalone_initial_high_level_goals,
    initial_low_level_goals=standalone_initial_low_level_goals,
    regenerate_low_level_goals=regenerate_low_level_goals_for_standalone_cycle,
    evaluation_output_directory=standalone_iterations_dir,
    max_iterations=STANDALONE_MAX_ITERATIONS,
)

print(f"Converged: {standalone_cycle_result.converged}")
print(f"Stop reason: {standalone_cycle_result.stop_reason}")
print(
    "Completed iterations: "
    f"{standalone_cycle_result.completed_iterations}/{standalone_cycle_result.max_iterations}"
)
print(f"High-level goals added during the cycle: {len(standalone_cycle_result.added_high_level_goals)}")
for added_goal in standalone_cycle_result.added_high_level_goals:
    print(f"  + {added_goal.name}: {added_goal.description}")

if standalone_cycle_result.unresolved_bottom_up_errors:
    print("Unresolved bottom-up errors:", standalone_cycle_result.unresolved_bottom_up_errors)
if standalone_cycle_result.unresolved_global_evaluation_errors:
    print("Unresolved global-evaluation errors:", standalone_cycle_result.unresolved_global_evaluation_errors)
if standalone_cycle_result.unresolved_empty_branches:
    print("Unresolved empty branches:", standalone_cycle_result.unresolved_empty_branches)
if standalone_cycle_result.unresolved_documentation_coverage_error:
    print(
        "Unresolved documentation-coverage error:",
        standalone_cycle_result.unresolved_documentation_coverage_error,
    )

Low Level Goals STARTING... (attempt 1)
No feedback provided!
This is the provided sys prompt:  You are a helpful assistant expert in software engineering tasks. Elicit low-level goals for a specific stakeholder in a software project. The low-level goals that you create MUST be structured to match against a set of API calls. Don't be too generic, for example, avoid goals like 'make the software fast', 'develop a web interface' etc.Each low-level goal MUST be phrased as an interaction with the system that could be implemented via an API call.Avoid generic goals. Instead, break them down into atomic actions linked to system capabilities. Following the Goal-Oriented Requirements Engineering (GORE) framework, low-level goals are technical objectives that describe 'how' the high-level goals will be achieved. 
They are more concrete and are eventually refined into specific requirements or software specifications. 
Focus: Implementation and constraints. Generate ONLY the functional goals.
Low

In [33]:
print("=== Comparison: top-down baseline vs. after the bottom-up cycle ===\n")

print(
    "High-level goals — baseline: "
    f"{len(standalone_initial_high_level_goals.goals)} | "
    f"after bottom-up: {len(standalone_cycle_result.final_high_level_goals.goals)}"
)
print(
    "Low-level goals  — baseline: "
    f"{len(standalone_initial_low_level_goals.low_level_goals)} | "
    f"after bottom-up: {len(standalone_cycle_result.final_low_level_goals.low_level_goals)}\n"
)

print("--- High-level goals: baseline (top-down only) ---")
for goal in standalone_initial_high_level_goals.goals:
    print(f"- [{goal.actor.name}] {goal.name}: {goal.description}")

print("\n--- High-level goals: after the bottom-up cycle ---")
for goal in standalone_cycle_result.final_high_level_goals.goals:
    print(f"- [{goal.actor.name}] {goal.name}: {goal.description}")

# Per-branch decisions from the last executed iteration: this is what tells
# us whether the bottom-up reconstruction agreed with the top-down parent
# (CONFIRM_BRANCH) or found a reason to revise/replace/add a goal.
last_iteration = standalone_cycle_result.iterations[-1]
print(f"\n--- Per-branch decisions (iteration {last_iteration.iteration}) ---")
for branch_id, evaluation in last_iteration.global_evaluations.items():
    print(
        f"- {branch_id}: {evaluation.decision} "
        f"— parent: {evaluation.original_high_level_goal.name}"
    )

decision_counts = {}
for evaluation in last_iteration.global_evaluations.values():
    decision_counts[evaluation.decision] = decision_counts.get(evaluation.decision, 0) + 1
print("\nDecision counts:", decision_counts)

=== Comparison: top-down baseline vs. after the bottom-up cycle ===

High-level goals — baseline: 5 | after bottom-up: 6
Low-level goals  — baseline: 17 | after bottom-up: 28

--- High-level goals: baseline (top-down only) ---
- [Citizen] HLG_001: The citizen aims to easily report urban issues and inconveniences in their environment, ensuring their concerns are addressed by the municipality and can track the status of their reports throughout the resolution process, including receiving updates on changes to their reports over time.
- [Municipal Operator] HLG_002: The municipal operator seeks to efficiently review, manage, and respond to citizen reports to ensure timely resolutions of urban issues, while providing clear communication to citizens regarding the status of their reports and maintaining data accuracy and transparency.
- [External Maintenance Personnel] HLG_003: The external maintenance personnel aim to receive timely notifications for assigned reports and communicate effecti

### 7.5 How to read this comparison

If most branches show `CONFIRM_BRANCH`, the top-down hierarchy was already consistent with what can be reconstructed bottom-up from its own low-level goals — the bottom-up extension found nothing to improve for those branches.

Any other decision points to a place where the bottom-up cycle detected a mismatch and therefore a potential improvement over the top-down-only baseline:
- `REGENERATE_LOW_LEVEL_GOALS`: the parent high-level goal only partially covers its low-level goals.
- `MATCHES_OTHER_HIGH_LEVEL_GOAL`: the branch was actually pursuing a different high-level goal already present elsewhere.
- `ADD_NEW_HIGH_LEVEL_GOAL`: an autonomous functional intention was missing from the original high-level-goal collection.
- `REWRITE_ORIGINAL_HIGH_LEVEL_GOAL`: the original high-level goal itself was incorrect or unsupported by the documentation.

The higher the proportion of non-`CONFIRM_BRANCH` decisions (and the more high-level goals added through documentation-coverage checks), the more the bottom-up extension changed — and arguably improved — the goal hierarchy compared to the top-down-only baseline.

In [34]:
standalone_output_file = OUTPUT_PATH / f"{baseline_output_file.stem}_bottom_up_standalone.json"

with standalone_output_file.open("w", encoding="utf-8") as file:
    json.dump(
        {
            "baseline_source_file": baseline_output_file.name,
            "globalGoalCycle": standalone_cycle_result.model_dump(mode="json"),
            "finalHighLevelGoals": standalone_cycle_result.final_high_level_goals.model_dump(
                mode="json"
            ),
            "finalLowLevelGoals": standalone_cycle_result.final_low_level_goals.model_dump(
                mode="json"
            ),
        },
        file,
        indent=4,
        ensure_ascii=False,
    )

print(f"Standalone bottom-up result saved to: {standalone_output_file}")

Standalone bottom-up result saved to: c:\Users\agnes_lryeu3v\Desktop\tesi\output\SIA Project 25 26_ONE_SHOT_bottom_up_standalone.json
